# Kaggle launcher (THIN) — two-detector comparison

Pulls a pinned commit of [Nafish32/pulmonary](https://github.com/Nafish32/pulmonary) and runs `src.pipeline.run_all` for each detector, then `compare_checkpoints.py`. No business logic here.

**Before running:**
- **Add data** (right panel): attach the **RSNA** dataset (`stage_2_train_labels.csv` + `stage_2_train_images/`). Attach **VinDr/VinBigData** too for the external-validation + recalibration stage (skips cleanly if absent).
- Settings: **GPU on** (single T4 is fine — both configs use `device:"0"`; Kaggle multi-GPU DDP breaks checkpoint-resume), **Internet on**, **Persistence: Variables and Files** so a killed run resumes and `keep/` survives session restarts.

**How to run (NOT a blind "Run All"):**
1. Cells 1–2 always (clone + setup helper). 3 = pre-flight (dataset + GPU).
2. Run the detector cell(s) you want: **rtdetr** and/or **yolo26m**. Each is a full ~12 hr `run_all` producing every trust-package number. Killed mid-train? Re-run the **same** cell — it resumes from the last checkpoint. A finished detector's cell **skips** (guards on the stashed `best.pt`), so no accidental retrain.
3. Comparison cell runs once **both** `best.pt` exist; otherwise it skips with a note.

Both configs are `fast_mode:false` → they share the `train_thesis` checkpoint slot. `train_one` moves each finished run aside to `train_thesis_<label>` so the other detector always starts **fresh**, in either order.

**"Later" caveat:** `keep/` survives session *restarts* of this notebook, **not** a brand-new container days later. If you train the second detector in a fresh container, save the first `*_best.pt` as a Kaggle output dataset and re-attach it.

The **Eval-only** cell at the bottom re-scores an existing `best.pt` without retraining (~7 min).

In [ ]:
# 1) Pull pinned commit + install (idempotent: safe to re-run in the same session)
import os, subprocess

REPO_URL = "https://github.com/Nafish32/pulmonary.git"
COMMIT   = "main"   # track latest. Pin to a SHA (e.g. "da62685") to freeze a thesis-numbers run.
REPO_DIR = "/kaggle/working/repo"

# CRITICAL: step out of REPO_DIR before deleting it. A prior run left the process
# CWD inside REPO_DIR; rm -rf on the CWD gives 'getcwd: cannot access parent directories'
# and every shell command after that silently no-ops.
os.chdir("/kaggle/working")

def sh(cmd):
    print(">", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd="/kaggle/working")

sh(f"rm -rf {REPO_DIR}")
sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {COMMIT}")
sh(f"pip install -q -r {REPO_DIR}/requirements.txt")
sh(f"pip install -q -e {REPO_DIR}")

# print the exact SHA that ran, so any number is traceable even when COMMIT='main'
sha = subprocess.check_output(f"git -C {REPO_DIR} rev-parse HEAD", shell=True).decode().strip()
print("[GIT] running commit:", sha)

In [ ]:
# 2) Setup + reusable trainer (validates both configs up front, prints detectors)
import os, logging, sys, shutil, subprocess
os.chdir("/kaggle/working/repo")
logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)  # src logs visible + live

# Drop stale src.* modules: Cell 1 re-clones the repo, but a re-run without a
# kernel restart keeps old code cached in sys.modules (traceback shows new file
# lines while the OLD code object runs). Pop them so imports below load fresh.
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    sys.modules.pop(m)

from src.config.loader import load_config
from src.pipeline import run_all

KEEP = "/kaggle/working/keep"; os.makedirs(KEEP, exist_ok=True)
# BOTH configs are fast_mode=false -> they share this one checkpoint slot.
SLOT = "/kaggle/working/outputs/runs/train_thesis"

def train_one(cfg_path, label):
    """Full run_all for one detector, then stash best.pt + results.md and move the
    shared slot aside so the OTHER detector starts fresh (order-independent).

    Idempotent: a finished detector (its {label}_best.pt exists) is skipped, so a
    re-run / 'Run All' never retrains it. Killed mid-train -> best.pt not yet
    written -> re-running resumes from the slot's last.pt.
    """
    best = f"{KEEP}/{label}_best.pt"
    if os.path.exists(best):
        print(f"[skip] {label} already done -> {best}"); return best
    cfg = load_config(cfg_path)
    print(f"train {label}:", cfg.detector_model_name, "| epochs", cfg.epochs,
          "| device", repr(cfg.device))
    run_all(cfg)                                       # ~12hr. Killed? re-run this cell -> resumes.
    shutil.copy(f"{SLOT}/weights/best.pt", best)
    shutil.copy("/kaggle/working/outputs/results.md", f"{KEEP}/{label}_results.md")
    shutil.move(SLOT, f"{SLOT}_{label}")               # free the shared slot for the other detector
    print(f"[done] {label} -> {best}"); return best

# Validate both configs now (fail fast on a bad config, not 5 min into a run) + confirm setup.
for _p in ("configs/rtdetr.yaml", "configs/thesis.yaml"):
    _c = load_config(_p)
    print(f"  {_p}: {_c.detector_model_name} | fast_mode={_c.fast_mode} epochs={_c.epochs} device={_c.device!r}")
print("setup OK | cwd:", os.getcwd(), "| keep/:", sorted(os.listdir(KEEP)), "| train_one() ready")

In [ ]:
# 3) Pre-flight check (fail fast BEFORE a ~12hr train): dataset found + GPUs visible.
# input_root is identical across configs, so either works for this check.
_cfg = load_config("configs/rtdetr.yaml")
from src.data.discover import discover_datasets
ds = discover_datasets(_cfg.input_root); print("DISCOVER OK", ds)  # raises here if RSNA missing

import pandas as pd
df = pd.read_csv(ds["rsna_csv"]); print("rows", len(df), "patients", df.patientId.nunique())

import torch
print("cuda:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
# 0 GPUs -> Accelerator not actually on. Both configs pin device:"0" (single T4/P100).

In [ ]:
# 4) Train RT-DETR (transformer detector -- the generality check).
# First stages (discover rglob, DICOM caching) are IO-bound and can run minutes
# with near-zero CPU/GPU and little output -- that is normal, not a hang.
# Watch for "loaded detector: rtdetr-l.pt" then epoch 1 (batch=4; OOM-retry to imgsz 512).
train_one("configs/rtdetr.yaml", "rtdetr")

In [ ]:
# 5) Train yolo26m (primary detector). Run now for both, or LATER when you want it.
# Same slot as rtdetr -> train_one already moved rtdetr's run aside, so this starts fresh.
train_one("configs/thesis.yaml", "yolo26m")

In [ ]:
# 6) Comparison table -- runs only once BOTH best.pt exist (else skips, so Run All
# doesn't hard-error while a detector is still training). Same eval_from_weights
# path as every other eval here -> numbers directly comparable. Adds param count +
# per-image inference latency. Per-model trust packages are in keep/*_results.md.
YOLO_BEST, RTDETR_BEST = f"{KEEP}/yolo26m_best.pt", f"{KEEP}/rtdetr_best.pt"
if os.path.exists(YOLO_BEST) and os.path.exists(RTDETR_BEST):
    OUT = f"{KEEP}/comparison.md"
    subprocess.run(["python", "compare_checkpoints.py", OUT,
                    f"yolo26m=configs/thesis.yaml:{YOLO_BEST}",
                    f"rtdetr=configs/rtdetr.yaml:{RTDETR_BEST}"], check=True)
    from IPython.display import Markdown, display
    display(Markdown(open(OUT).read()))
else:
    print("comparison skipped: need BOTH best.pt. keep/ has:", sorted(os.listdir(KEEP)))

## Eval-only (recovery / re-score) — manual, commented so it stays skipped

Re-score an existing `best.pt` **without** the ~12 hr train (same data prep + all trust stages minus ensemble spread, ~7 min). Use it to recover a run whose training finished but eval was wrong, or to score arbitrary weights. Requires cells 1–2 first. Pick the config that matches the checkpoint's detector family. **Uncomment to use.**

In [ ]:
# from src.pipeline import eval_from_weights
# from IPython.display import Markdown, display
#
# # MUST match the checkpoint's family: rtdetr.yaml for rtdetr_best.pt, thesis.yaml for yolo26m_best.pt.
# cfg = load_config("configs/rtdetr.yaml")
# WEIGHTS = f"{KEEP}/rtdetr_best.pt"                 # stashed by train_one
# # or a still-in-slot run:  WEIGHTS = f"{SLOT}_rtdetr/weights/best.pt"
# # or an uploaded weights dataset:  WEIGHTS = "/kaggle/input/<your-best-pt-dataset>/best.pt"
#
# results_md = eval_from_weights(cfg, WEIGHTS)
# print("results:", results_md)
# display(Markdown(open(results_md).read()))